# **<u> Does an increase in hours spent on social media cause a decrease in academic performance GPA?**

In [1]:
# Data Analysis packages
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Statistics Packages
import scipy.stats as stats
import statsmodels.api as sm

### **Dataset Used:**

https://www.kaggle.com/datasets/muqniturrehman/social-media-and-academic-performance-of-students

### **Dataset Description:**
- Unfortunatly, there is no description given for this dataset on kaggle.
- So, we will assume that the sample of college/university students was collected randomly.
- Random sampling did in fact take place, but as for the treatment we will assume it was not assigned at random.
- The students choose to interact with social media on their own terms, for however long they wanted.

### **Innitial Assumption:**
- Does an increase in hours spent on social media cause a decrease in academic performance GPA?

### **Linear Regression for Causal Inferrence:**

> $y$ = $\beta_0$ + $\beta_1$ $D_i$ + $\epsilon_{it}$

$y$ = The outcome variable. The dependent variable.

$\beta_0$ = The outcome of the control group.

$\beta_1$ = The average difference in the change of the outcome between the treated and controlled groups.

$D_i$ = The treatment variable, The independent variable. When $D_i$ = 1 = treated, when $D_i$ = 0 = control.

$\epsilon_i$ = The error term. Includes all unidentified cofounder variables.

### **Outcome Variable:**
- Student GPA.
- In the dataset this variable is categorical, and students can fall into buckets of ranges such as 3.5-4.0, 3.0-3.5, and 2.5-3.0.
- We will modify this and take the upper bound of these ranges to make the data integer and continuous for the linear regression output.
- So, if the range is 3.5-4.0, we will take 4.0. If the range is 3.0-3.5, we will take 3.5. If the range is 2.5-3.0, we will take 3.0.

### **Treatment Variable:**
- Hours on Social Media.
- Similar to the GPA column, the dataset treats hours on social media as a categorical variable, which contains ranges such as 3-4 hours, 1-2 hours, more than 6 hours, and less than 1 hour.
- We want an integer continuous value, so we will take the upper bound of these ranges.
- If the range is less than 1 hour, we will take 1. If the range is 1-2 hours, we will take 2. If the range is 3-4 hours, we will take 4, and if the rang is greater than 6 hours, we will take 7 (This is because there is a range 5-6, and we will take 6 for that).
- Since the treatment is continuous, there does not exist a control group, so the average treatment effect is interpreted as the change in student GPA from one additional hour of social media use.

### **Possible Cofounders:**
- Age: This variable is in categorical ranges, so we must convert to dummy variables.
- platforms: We must convert this variable to individual dummy variables.
- Divice: Must be converted to dummy variables.
- Affects Performance: This is the personal assumption people have about their use of social media, and if they think it affects them academically.
- Academic Use Frequency: Dummy variables which state how frequently students use social media for academic purposes.

### **Linear Regression Equation for Problem:**
> Student GPA = $\beta_0$ + $\beta_1$ (Hours Spent on Social Media) + $\beta_{k-1}$ (k-1 Age Ranges) + $\beta_{q-1}$ (q-1 Social Media Platforms) + $\beta_{i-1}$ (i-1 Divices Used) + $\beta_2$ (Affects Performance) + $\beta_{j-1}$ (j-1 Academic Use Frequency) + $\epsilon$

---

## **Loading and Cleaning Data:**

In [2]:
df = pd.read_csv("C:\BROCK U COURSE DOCS\DataSets\Effect of Social Media On Academic Performance of Students - realistic_social_media_academic_impact-selected-columns.csv")
df

,Age,Grade Level,Hours on Social Media,Platforms,Device,Reasons,GPA,Affects Performance,Effect Details,Academic Use Frequency
0,16-18,College/University,More than 6 hours,"Instagram, Snapchat, WhatsApp",Smartphone,Educational purposes,3.5-4.0,Yes,Distracts me from studying,Frequently
1,16-18,College/University,3-4 hours,"Instagram, WhatsApp",Smartphone,News and updates,3.0-3.5,Yes,"Helps me with research and study groups, Distr...",Occasionally
2,16-18,College/University,3-4 hours,"Instagram, WhatsApp",Smartphone,Socializing with friends,2.5-3.0,Yes,"Helps me with research and study groups, Cause...",Occasionally
3,16-18,College/University,More than 6 hours,Instagram,Smartphone,"News and updates, Professional networking, Ent...",3.0-3.5,No,"Motivates me through educational content, Caus...",Frequently
4,Over 21,College/University,1-2 hours,YouTube,Laptop,"Professional networking, Entertainment, News a...",2.5-3.0,Yes,"Causes me to procrastinate, Motivates me throu...",Frequently
...,...,...,...,...,...,...,...,...,...,...
295,19-21,College/University,1-2 hours,"Facebook, Instagram, WhatsApp",Smartphone,Professional networking,3.5-4.0,Yes,"Motivates me through educational content, Help...",Occasionally
296,19-21,College/University,3-4 hours,"Facebook, Instagram, WhatsApp",Smartphone,"Professional networking, Educational purposes,...",3.0-3.5,Yes,"Causes me to procrastinate, Distracts me from ...",Occasionally
297,19-21,College/University,5-6 hours,"WhatsApp, YouTube",Laptop,Socializing with friends,3.0-3.5,Yes,Helps me with research and study groups,Frequently
298,19-21,College/University,1-2 hours,"Facebook, Instagram, Twitter, Snapchat, TikTok...",Smartphone,"Professional networking, Educational purposes",2.5-3.0,No,Causes me to procrastinate,Frequently


In [3]:
df.duplicated().sum()

np.int64(0)

> No duplicated rows to be deleted.

In [4]:
df.isna().sum()

Age                       0
Grade Level               0
Hours on Social Media     0
Platforms                 0
Device                    0
Reasons                   0
GPA                       0
Affects Performance       0
Effect Details            0
Academic Use Frequency    0
dtype: int64

> No NaN values that should be deleted.

In [5]:
age = pd.get_dummies(df['Age'], drop_first= True).astype(int)

df = pd.merge(df, age, left_index= True, right_index= True)

df = df.drop(labels= 'Age', axis= 1)

df

,Grade Level,Hours on Social Media,Platforms,Device,Reasons,GPA,Affects Performance,Effect Details,Academic Use Frequency,19-21,Over 21
0,College/University,More than 6 hours,"Instagram, Snapchat, WhatsApp",Smartphone,Educational purposes,3.5-4.0,Yes,Distracts me from studying,Frequently,0,0
1,College/University,3-4 hours,"Instagram, WhatsApp",Smartphone,News and updates,3.0-3.5,Yes,"Helps me with research and study groups, Distr...",Occasionally,0,0
2,College/University,3-4 hours,"Instagram, WhatsApp",Smartphone,Socializing with friends,2.5-3.0,Yes,"Helps me with research and study groups, Cause...",Occasionally,0,0
3,College/University,More than 6 hours,Instagram,Smartphone,"News and updates, Professional networking, Ent...",3.0-3.5,No,"Motivates me through educational content, Caus...",Frequently,0,0
4,College/University,1-2 hours,YouTube,Laptop,"Professional networking, Entertainment, News a...",2.5-3.0,Yes,"Causes me to procrastinate, Motivates me throu...",Frequently,0,1
...,...,...,...,...,...,...,...,...,...,...,...
295,College/University,1-2 hours,"Facebook, Instagram, WhatsApp",Smartphone,Professional networking,3.5-4.0,Yes,"Motivates me through educational content, Help...",Occasionally,1,0
296,College/University,3-4 hours,"Facebook, Instagram, WhatsApp",Smartphone,"Professional networking, Educational purposes,...",3.0-3.5,Yes,"Causes me to procrastinate, Distracts me from ...",Occasionally,1,0
297,College/University,5-6 hours,"WhatsApp, YouTube",Laptop,Socializing with friends,3.0-3.5,Yes,Helps me with research and study groups,Frequently,1,0
298,College/University,1-2 hours,"Facebook, Instagram, Twitter, Snapchat, TikTok...",Smartphone,"Professional networking, Educational purposes",2.5-3.0,No,Causes me to procrastinate,Frequently,1,0


> We removed the categorical age column for dummy variable columns. One is for ages 19-21, and the other is over 21. If both are 0, then age falls within the range 16-18.

In [6]:
df.columns

Index(['Grade Level', 'Hours on Social Media', 'Platforms', 'Device',
       'Reasons', 'GPA', 'Affects Performance', 'Effect Details',
       'Academic Use Frequency', '19-21', 'Over 21'],
      dtype='object')

In [7]:
df = df.drop(labels= ['Grade Level', 'Reasons', 'Effect Details'], axis= 1)
df

,Hours on Social Media,Platforms,Device,GPA,Affects Performance,Academic Use Frequency,19-21,Over 21
0,More than 6 hours,"Instagram, Snapchat, WhatsApp",Smartphone,3.5-4.0,Yes,Frequently,0,0
1,3-4 hours,"Instagram, WhatsApp",Smartphone,3.0-3.5,Yes,Occasionally,0,0
2,3-4 hours,"Instagram, WhatsApp",Smartphone,2.5-3.0,Yes,Occasionally,0,0
3,More than 6 hours,Instagram,Smartphone,3.0-3.5,No,Frequently,0,0
4,1-2 hours,YouTube,Laptop,2.5-3.0,Yes,Frequently,0,1
...,...,...,...,...,...,...,...,...
295,1-2 hours,"Facebook, Instagram, WhatsApp",Smartphone,3.5-4.0,Yes,Occasionally,1,0
296,3-4 hours,"Facebook, Instagram, WhatsApp",Smartphone,3.0-3.5,Yes,Occasionally,1,0
297,5-6 hours,"WhatsApp, YouTube",Laptop,3.0-3.5,Yes,Frequently,1,0
298,1-2 hours,"Facebook, Instagram, Twitter, Snapchat, TikTok...",Smartphone,2.5-3.0,No,Frequently,1,0


> Dropped unimportant columns which will not be used in the regression or analysis.

In [8]:
df['Hours on Social Media'].value_counts()

Hours on Social Media
3-4 hours            75
5-6 hours            68
More than 6 hours    67
1-2 hours            63
Less than 1 hour     27
Name: count, dtype: int64

In [9]:
df['Hours on Social Media'] = df['Hours on Social Media'].map({
    'Less than 1 hour': 1,
    '1-2 hours': 2,
    '3-4 hours': 4,
    '5-6 hours': 6,
    'More than 6 hours': 7
})

df

,Hours on Social Media,Platforms,Device,GPA,Affects Performance,Academic Use Frequency,19-21,Over 21
0,7,"Instagram, Snapchat, WhatsApp",Smartphone,3.5-4.0,Yes,Frequently,0,0
1,4,"Instagram, WhatsApp",Smartphone,3.0-3.5,Yes,Occasionally,0,0
2,4,"Instagram, WhatsApp",Smartphone,2.5-3.0,Yes,Occasionally,0,0
3,7,Instagram,Smartphone,3.0-3.5,No,Frequently,0,0
4,2,YouTube,Laptop,2.5-3.0,Yes,Frequently,0,1
...,...,...,...,...,...,...,...,...
295,2,"Facebook, Instagram, WhatsApp",Smartphone,3.5-4.0,Yes,Occasionally,1,0
296,4,"Facebook, Instagram, WhatsApp",Smartphone,3.0-3.5,Yes,Occasionally,1,0
297,6,"WhatsApp, YouTube",Laptop,3.0-3.5,Yes,Frequently,1,0
298,2,"Facebook, Instagram, Twitter, Snapchat, TikTok...",Smartphone,2.5-3.0,No,Frequently,1,0


> Converted the categorical ranges for hours spent on social media to continuous integer values.

In [10]:
df['Platforms'].value_counts()

Platforms
Instagram                                                   48
Instagram, Snapchat, WhatsApp                               46
Instagram, WhatsApp                                         44
Facebook, Instagram, Twitter, Snapchat, TikTok, WhatsApp    44
WhatsApp, YouTube                                           42
Facebook, Instagram, WhatsApp                               42
YouTube                                                     34
Name: count, dtype: int64

In [11]:
df['Instagram'] = df['Platforms'].str.contains('Instagram').astype(int)
df['Snapchat'] = df['Platforms'].str.contains('Snapchat').astype(int)
df['WhatsApp'] = df['Platforms'].str.contains('WhatsApp').astype(int)
df['Facebook'] = df['Platforms'].str.contains('Facebook').astype(int)
df['Twitter'] = df['Platforms'].str.contains('Twitter').astype(int)
df['TikTok'] = df['Platforms'].str.contains('TikTok').astype(int)
df['YouTube'] = df['Platforms'].str.contains('YouTube').astype(int)

df = df.drop(labels= 'Platforms', axis= 1)
df

,Hours on Social Media,Device,GPA,Affects Performance,Academic Use Frequency,19-21,Over 21,Instagram,Snapchat,WhatsApp,Facebook,Twitter,TikTok,YouTube
0,7,Smartphone,3.5-4.0,Yes,Frequently,0,0,1,1,1,0,0,0,0
1,4,Smartphone,3.0-3.5,Yes,Occasionally,0,0,1,0,1,0,0,0,0
2,4,Smartphone,2.5-3.0,Yes,Occasionally,0,0,1,0,1,0,0,0,0
3,7,Smartphone,3.0-3.5,No,Frequently,0,0,1,0,0,0,0,0,0
4,2,Laptop,2.5-3.0,Yes,Frequently,0,1,0,0,0,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
295,2,Smartphone,3.5-4.0,Yes,Occasionally,1,0,1,0,1,1,0,0,0
296,4,Smartphone,3.0-3.5,Yes,Occasionally,1,0,1,0,1,1,0,0,0
297,6,Laptop,3.0-3.5,Yes,Frequently,1,0,0,0,1,0,0,0,1
298,2,Smartphone,2.5-3.0,No,Frequently,1,0,1,1,1,1,1,1,0


> Created binary variables for the different social media platforms.

In [12]:
df['Device'].value_counts()

Device
Smartphone    221
Laptop         61
Tablet         18
Name: count, dtype: int64

In [13]:
devices = pd.get_dummies(df['Device'], drop_first= True).astype(int)

df = pd.merge(df, devices, left_index= True, right_index= True)

df = df.drop(labels= 'Device', axis= 1)

df

,Hours on Social Media,GPA,Affects Performance,Academic Use Frequency,19-21,Over 21,Instagram,Snapchat,WhatsApp,Facebook,Twitter,TikTok,YouTube,Smartphone,Tablet
0,7,3.5-4.0,Yes,Frequently,0,0,1,1,1,0,0,0,0,1,0
1,4,3.0-3.5,Yes,Occasionally,0,0,1,0,1,0,0,0,0,1,0
2,4,2.5-3.0,Yes,Occasionally,0,0,1,0,1,0,0,0,0,1,0
3,7,3.0-3.5,No,Frequently,0,0,1,0,0,0,0,0,0,1,0
4,2,2.5-3.0,Yes,Frequently,0,1,0,0,0,0,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
295,2,3.5-4.0,Yes,Occasionally,1,0,1,0,1,1,0,0,0,1,0
296,4,3.0-3.5,Yes,Occasionally,1,0,1,0,1,1,0,0,0,1,0
297,6,3.0-3.5,Yes,Frequently,1,0,0,0,1,0,0,0,1,0,0
298,2,2.5-3.0,No,Frequently,1,0,1,1,1,1,1,1,0,1,0


> Added dummy variable for devices. If both smartphone and tablet columns are 0, then device of choice is laptop.

In [14]:
df['GPA'].value_counts()

GPA
3.0-3.5    96
3.5-4.0    84
2.5-3.0    79
2.0-2.5    41
Name: count, dtype: int64

In [15]:
df['GPA'] = df['GPA'].map({
    '2.0-2.5': 2.5,
    '2.5-3.0': 3.0,
    '3.0-3.5': 3.5,
    '3.5-4.0': 4.0
})

df

,Hours on Social Media,GPA,Affects Performance,Academic Use Frequency,19-21,Over 21,Instagram,Snapchat,WhatsApp,Facebook,Twitter,TikTok,YouTube,Smartphone,Tablet
0,7,4.0,Yes,Frequently,0,0,1,1,1,0,0,0,0,1,0
1,4,3.5,Yes,Occasionally,0,0,1,0,1,0,0,0,0,1,0
2,4,3.0,Yes,Occasionally,0,0,1,0,1,0,0,0,0,1,0
3,7,3.5,No,Frequently,0,0,1,0,0,0,0,0,0,1,0
4,2,3.0,Yes,Frequently,0,1,0,0,0,0,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
295,2,4.0,Yes,Occasionally,1,0,1,0,1,1,0,0,0,1,0
296,4,3.5,Yes,Occasionally,1,0,1,0,1,1,0,0,0,1,0
297,6,3.5,Yes,Frequently,1,0,0,0,1,0,0,0,1,0,0
298,2,3.0,No,Frequently,1,0,1,1,1,1,1,1,0,1,0


> Modified the GPA column by converting the categorical ranges to numerical continuous values.

In [16]:
df['Affects Performance'] = df['Affects Performance'].map({'Yes': 1, 'No': 0}).astype(bool).astype(int)

df

,Hours on Social Media,GPA,Affects Performance,Academic Use Frequency,19-21,Over 21,Instagram,Snapchat,WhatsApp,Facebook,Twitter,TikTok,YouTube,Smartphone,Tablet
0,7,4.0,1,Frequently,0,0,1,1,1,0,0,0,0,1,0
1,4,3.5,1,Occasionally,0,0,1,0,1,0,0,0,0,1,0
2,4,3.0,1,Occasionally,0,0,1,0,1,0,0,0,0,1,0
3,7,3.5,0,Frequently,0,0,1,0,0,0,0,0,0,1,0
4,2,3.0,1,Frequently,0,1,0,0,0,0,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
295,2,4.0,1,Occasionally,1,0,1,0,1,1,0,0,0,1,0
296,4,3.5,1,Occasionally,1,0,1,0,1,1,0,0,0,1,0
297,6,3.5,1,Frequently,1,0,0,0,1,0,0,0,1,0,0
298,2,3.0,0,Frequently,1,0,1,1,1,1,1,1,0,1,0


> Wow that was a mess. I dont know what data type that column was before but it DID NOT want to become binary. Anyways, I converted yes it does affect academic performance opinion to 1, and no it does not affect academic performance response to 0.

In [17]:
df['Academic Use Frequency'].value_counts()

Academic Use Frequency
Always          103
Frequently      101
Occasionally     96
Name: count, dtype: int64

In [18]:
df['Academic Use Freq Always'] = (df['Academic Use Frequency'] == 'Always').astype(int)
df['Academic Use Freq Frequently'] = (df['Academic Use Frequency'] == 'Frequently').astype(int)

df = df.drop(labels= 'Academic Use Frequency', axis= 1)

df

,Hours on Social Media,GPA,Affects Performance,19-21,Over 21,Instagram,Snapchat,WhatsApp,Facebook,Twitter,TikTok,YouTube,Smartphone,Tablet,Academic Use Freq Always,Academic Use Freq Frequently
0,7,4.0,1,0,0,1,1,1,0,0,0,0,1,0,0,1
1,4,3.5,1,0,0,1,0,1,0,0,0,0,1,0,0,0
2,4,3.0,1,0,0,1,0,1,0,0,0,0,1,0,0,0
3,7,3.5,0,0,0,1,0,0,0,0,0,0,1,0,0,1
4,2,3.0,1,0,1,0,0,0,0,0,0,1,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
295,2,4.0,1,1,0,1,0,1,1,0,0,0,1,0,0,0
296,4,3.5,1,1,0,1,0,1,1,0,0,0,1,0,0,0
297,6,3.5,1,1,0,0,0,1,0,0,0,1,0,0,0,1
298,2,3.0,0,1,0,1,1,1,1,1,1,0,1,0,0,1


> Converted the academic use frequency column to dummy variables. If both Always and Frequent are 0, then frequency is Occasionally.

## **Creating Linear Regression Model:**

In [19]:
df.columns

Index(['Hours on Social Media', 'GPA', 'Affects Performance', '19-21',
       'Over 21', 'Instagram', 'Snapchat', 'WhatsApp', 'Facebook', 'Twitter',
       'TikTok', 'YouTube', 'Smartphone', 'Tablet', 'Academic Use Freq Always',
       'Academic Use Freq Frequently'],
      dtype='object')

In [20]:
y = df['GPA']

x = df[['Hours on Social Media', 'Affects Performance', '19-21',
       'Over 21', 'Instagram', 'Snapchat', 'WhatsApp', 'Facebook', 'Twitter',
       'TikTok', 'YouTube', 'Smartphone', 'Tablet', 'Academic Use Freq Always',
       'Academic Use Freq Frequently']]

x = sm.add_constant(x)

model = sm.OLS(y, x).fit()
model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                    GPA   R-squared:                       0.048
Model:                            OLS   Adj. R-squared:                  0.004
Method:                 Least Squares   F-statistic:                     1.099
Date:                Mon, 17 Aug 2026   Prob (F-statistic):              0.360
Time:                        19:55:47   Log-Likelihood:                -214.00
No. Observations:                 300   AIC:                             456.0
Df Residuals:                     286   BIC:                             507.9
Df Model:                          13                                         
Covariance Type:            nonrobust                                         
================================================================================================
                                   coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------------
const                            2.1913      0.090     24.432      0.000       2.015       2.368
Hours on Social Media           -0.0060      0.015     -0.411      0.682      -0.035       0.023
Affects Performance             -0.1194      0.066     -1.807      0.072      -0.250       0.011
19-21                            0.0265      0.072      0.369      0.712      -0.115       0.168
Over 21                         -0.1123      0.098     -1.144      0.254      -0.305       0.081
Instagram                        1.1149      0.060     18.724      0.000       0.998       1.232
Snapchat                         0.0319      0.101      0.316      0.753      -0.167       0.231
WhatsApp                         0.1199      0.080      1.499      0.135      -0.038       0.277
Facebook                        -0.0478      0.104     -0.458      0.647      -0.253       0.157
Twitter                         -0.0279      0.075     -0.374      0.708      -0.175       0.119
TikTok                          -0.0279      0.075     -0.374      0.708      -0.175       0.119
YouTube                          1.0763      0.060     17.886      0.000       0.958       1.195
Smartphone                       0.0990      0.074      1.337      0.182      -0.047       0.245
Tablet                           0.2268      0.142      1.603      0.110      -0.052       0.505
Academic Use Freq Always         0.0270      0.073      0.372      0.711      -0.116       0.170
Academic Use Freq Frequently     0.0465      0.073      0.635      0.526      -0.098       0.191
==============================================================================
Omnibus:                       42.830   Durbin-Watson:                   2.087
Prob(Omnibus):                  0.000   Jarque-Bera (JB):               13.416
Skew:                          -0.227   Prob(JB):                      0.00122
Kurtosis:                       2.069   Cond. No.                     1.41e+17
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The smallest eigenvalue is 4.16e-31. This might indicate that there are
strong multicollinearity problems or that the design matrix is singular.
"""

## **Conclusion:**

### **Linear Regression Equation for Problem:**
> Student GPA = 2.1913 -0.0060 (Hours Spent on Social Media) + $\beta_{k-1}$ (k-1 Age Ranges) + $\beta_{q-1}$ (q-1 Social Media Platforms) + $\beta_{i-1}$ (i-1 Divices Used) -0.1194 (Affects Performance) + $\beta_{j-1}$ (j-1 Academic Use Frequency) + $\epsilon$

### **Hypothesis Test for Treatment Beta Coefficient Significance:**

Ho: Population Hours Spent on Social Media $\beta_1$ = 0

Ha: Population Hours Spent on Social Media $\beta_1$ != 0

t-stat = ($\beta_1$ - 0) / SE[$\beta_1$]

t-stat = (-0.0060 - 0) / 0.015

t-stat = -0.411

p-value = The probability of observing a test statistic as extreme as the one observed, given the null hypothesis is true.

p-value = 0.682

$\alpha$ = 0.05

=> p-value > $\alpha$

=> 0.682 > 0.05

=> We fail to reject the null hypothesis; Population Hours Spent on Social Media $\beta_1$ = 0

### **Confidence Interval for Treatment Beta Coefficient:**

Confidence Interval = [$\beta_1$ - (t[0.05] * SE), $\beta_1$ + (t[0.05] * SE)]

Confidence Interval = [-0.0060 - (1.96 * 0.015), -0.0060 + (1.96 * 0.015)]

Confidence Interval = [-0.035, 0.023]

=> Confidence interval does contain zero and therefore these results confirm the hypothesis test that $\beta_1$ = 0.

### **Description, Reasoning, and Summary:**
- The model does confirm the relationship proposed in the initial assumption, which states that an increase in hours spent on social media will lead to a decrease in academic performance GPA.
- The model shows that an increase in hours of social media use will decrease student GPA by an average of 0.0060 while controlling for age, social media platforms used, device used, personal oppinion on affect on acadeomic performance, and use of social media for academic purposes.
- However, the main issue is that this relationship is not statistically significant at all.
- The difference between the sample $\beta_1$ and zero, given the standard deviation, is low, which tells us that the sample $\beta_1$ is very close to and practically zero.
- Since this is the case, the probability of observing a test statistic as extreme as the one observed, given the null hypothesis is true is high.
- This is because we assume the null hypothesis is true, and that population $\beta_1$ is equal to zero, which is the case and the sample $\beta_1$ does confirm the null to be true.
- The sample $\beta_1$ is very close to zero, and it confirms the truth of the null hypothesis. The null hypothesis of the population $\beta_1$ accuratly represent the sample data.
- Therefore the null hypothesis has failed to be rejected. The population average change in the outcome $\beta_1$ is zero.
- Since there is no statistical significance of the relationship between hours of social media use and GPA, there cannot possibly be any evidence for a causal effect.
- The sample data is collected at random, however the treatment is not assigned randomly since students choose to use social media on their own.
- Since the treatment is not random and the relationship is not statistically significant, then there must be other cofounders affecting the outcome and treatment variable which we are not controlling for.
- We cannot conclude that this relationship is causal, and we cannot conclude the relationship is significant.
- There does not seem to be any relatioship between hours of social media use and academic performance measured as GPA since the average treatment effect ATE, or the average change in the outcome (GPA) from one additional unit of the treatment (hours of social media use) is zero ($\beta_1$ = 0)
- Infact, all other independent variables seem to be completely insignificant, except for Youtube for some reason, which increases GPA by an average of 1.0763 due to one additional hour of use.
- The reason this may be the case is because people use youtube for academic purposes, such as watching tutorial videos related to some subject like statistics or data science.

In [21]:
df[df['YouTube'] == 1][['YouTube', 'GPA']].value_counts().reset_index()

,YouTube,GPA,count
0,1,3.0,26
1,1,3.5,26
2,1,4.0,16
3,1,2.5,8


- As seen here, students who use Youtube have very high GPAs.
- 26 students who use Youtube have a 3.0 GPA, another 26 have a 3.5, 16 have 4.0, and lastly 8 students have 2.5.

In [22]:
df[df['YouTube'] == 1][['YouTube', 'GPA']].value_counts(normalize= True).reset_index()

,YouTube,GPA,proportion
0,1,3.0,0.342105
1,1,3.5,0.342105
2,1,4.0,0.210526
3,1,2.5,0.105263


- Another way to view the data is that 34% of those students who use youtube have a 2.0 GPA, 34% have 3.5 GPA, and 21% have a 4.0 GPA.